# Modelagem Supervisionada: Decision Tree vs. Random Forest

**Trilha B — Predição de Qualidade do Ar Inadequada para Exercícios ao Ar Livre em São Paulo (Horizonte 24h)**

Este notebook apresenta o treinamento, a calibração de ponto de operação e o teste comparativo de dois algoritmos fundamentais de aprendizado de máquina supervisionado:
1. **Decision Tree (Árvore de Decisão)**: Modelo baseado em regras explícitas e alta interpretabilidade.
2. **Random Forest (Floresta Aleatória)**: Ensemble de 150 árvores com bagging e aleatorização de atributos para reduzir a variância.

### Critérios e Valores de Referência para Aprovação no Teste:
- **Custo Crítico (RFC):** O **Falso Negativo (FN)** é o erro de maior custo, pois afirmar que o ar está adequado quando está impróprio expõe praticantes de exercícios a riscos severos de saúde.
- **Piso 1 - DummyClassifier (Acurácia Ingênua):** 91,41% de acurácia, mas **Recall = 0%** e **299 FNs**.
- **Piso 2 - Persistência Temporal (Regra Operacional):** **Recall = 48,49%**, **F1 = 48,25%** e **154 FNs**.
- **Critério de Aprovação:** Para justificar a complexidade dos modelos de ML, o modelo deve superar a Persistência Temporal, alcançando **Recall > 48,49%** e **FN < 154**.

## 1. Importações e Configuração do Ambiente

In [ ]:
from pathlib import Path
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, Image

raiz = Path.cwd().resolve()
if raiz.name == "notebooks":
    raiz = raiz.parent

sys.path.insert(0, str(raiz / "src"))
print("Diretório raiz do projeto:", raiz)

## 2. Execução Completa da Modelagem

O script `src/modelos_arvore_random_forest.py` executa o pipeline unificado:
- Carregamento dos dados com purga de 24h na fronteira do treino (evitando vazamento de dados futuros);
- Validação temporal interna para calibrar o ponto de operação por $F_1$ e $F_2$ (com peso dobro no Recall);
- Treinamento final no conjunto completo de treino;
- Teste avaliado uma única vez no conjunto de teste (3.480 horas);
- Exportação dos pipelines em `.joblib` e gravação dos relatórios e gráficos.

In [ ]:
import modelos_arvore_random_forest as mod
mod.main()

## 3. Quadro Comparativo e Status de Aprovação no Teste

A tabela abaixo compara o desempenho dos baselines contra a Decision Tree e o Random Forest:

In [ ]:
tabela = pd.read_csv(raiz / "reports/modelos_decision_tree_random_forest/comparativo_completo_decision_tree_vs_random_forest.csv")
display(tabela)

## 4. Matrizes de Confusão: Decision Tree vs. Random Forest

Observe especialmente a célula de **Falso Negativo (FN)**, no canto inferior esquerdo de cada matriz (Classe Real 1 vs. Prevista 0):

In [ ]:
rep = raiz / "reports/modelos_decision_tree_random_forest"

print("=== DECISION TREE ===")
display(Image(filename=str(rep / "decision_tree/matriz_decision_tree_padrao.png")))

print("=== RANDOM FOREST (Calibrado em F2 para contenção de FN) ===")
display(Image(filename=str(rep / "random_forest/matriz_random_forest_calibrado_f2.png")))

## 5. Importância de Atributos: Diferenciação entre os Algoritmos

Quais features foram mais importantes para cada modelo?

In [ ]:
print("--- Top 5 Atributos: Decision Tree ---")
dt_imp = pd.read_csv(rep / "decision_tree/importancia_features_decision_tree.csv")
display(dt_imp.head(5))
display(Image(filename=str(rep / "decision_tree/importancia_features_decision_tree.png")))

print("--- Top 5 Atributos: Random Forest ---")
rf_imp = pd.read_csv(rep / "random_forest/importancia_features_random_forest.csv")
display(rf_imp.head(5))
display(Image(filename=str(rep / "random_forest/importancia_features_random_forest.png")))

## 6. Síntese dos Resultados e Conclusão

- **Decision Tree:** Excelente para regras interpretáveis e obteve o menor número absoluto de Falsos Negativos (**112 FNs**, Recall de **62,54%**), mas com maior taxa de alarmes falsos (378 FPs).
- **Random Forest:** Apresenta maior estabilidade estatística e estimativas de probabilidade suaves. Com limiar calibrado em $F_2 = 0,35$, alcançou **Recall de 55,18%** e **134 FNs**, superando a Persistência Temporal em todos os quesitos relevantes.